# 00 — Master Pipeline: AI Equity Research Lab

**FGV EAESP** — Pipeline completo (Aulas 1–8)

Este notebook executa **todo o pipeline** de ponta a ponta, da ingestao de dados ate a geracao do relatorio PDF final.

Cada celula e independente: pode ser re-executada individualmente (carrega dados de parquet quando necessario).

## Setup

In [1]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

import pandas as pd
pd.set_option("display.float_format", "{:.2f}".format)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: C:\Users\João Paulo\equity_research
Processed dir: C:\Users\João Paulo\equity_research\data\processed


## Aula 1–2: Ingestao de Dados

Coleta dados do CVM (DFP), Yahoo Finance (precos) e BCB/SGS (macro).

**Saidas:** `fundamentals_long.parquet`, `income_long.parquet`, `prices_daily.parquet`, `macro_monthly.parquet`

In [2]:
t0 = time.time()

# Ingest modules (skip if parquets already exist)
if not (PROCESSED_DIR / "prices_daily.parquet").exists():
    from src.ingest.cvm import run as run_cvm
    from src.ingest.market import run as run_market
    from src.ingest.macro import run as run_macro
    run_cvm()
    run_market()
    run_macro()
    print("Ingest pipeline executed.")
else:
    print("Parquets already exist, skipping ingest.")

# Verify
for f in ["fundamentals_long", "income_long", "prices_daily", "macro_monthly"]:
    df = pd.read_parquet(PROCESSED_DIR / f"{f}.parquet")
    print(f"  {f}: {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nAula 1-2 concluida em {time.time()-t0:.1f}s")

Parquets already exist, skipping ingest.
  fundamentals_long: 15939 rows x 7 cols
  income_long: 49 rows x 6 cols


  prices_daily: 16434 rows x 7 cols
  macro_monthly: 72 rows x 4 cols

Aula 1-2 concluida em 0.2s


## Aula 3: Engenharia de KPIs

Calcula 14 indicadores financeiros por empresa/ano e gera score composto 0–100.

**Saidas:** `kpis_wide.parquet`, `scores_2024.parquet`

In [3]:
t0 = time.time()

if not (PROCESSED_DIR / "kpis_wide.parquet").exists():
    from src.features.kpis import run as run_kpis
    from src.features.score import run as run_score
    run_kpis()
    run_score()
    print("KPI pipeline executed.")
else:
    print("KPI parquets already exist.")

kpis = pd.read_parquet(PROCESSED_DIR / "kpis_wide.parquet")
print(f"  kpis_wide: {kpis.shape[0]} rows x {kpis.shape[1]} cols")
print(f"  Years: {sorted(kpis['year'].unique())}")
print(f"  Tickers: {sorted(kpis['ticker'].unique())}")

print(f"\nAula 3 concluida em {time.time()-t0:.1f}s")

KPI parquets already exist.
  kpis_wide: 50 rows x 17 cols
  Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  Tickers: ['ABCB4', 'BBAS3', 'BBDC4', 'CMIG4', 'CPFE3', 'EGIE3', 'EQTL3', 'ITUB4', 'SANB11', 'TAEE11']

Aula 3 concluida em 0.0s


## Aula 4: NLP e Sentimento

Coleta textos via Google News RSS, analisa sentimento com DistilBERT, e gera textual index 0–100.

**Saidas:** `sentiment_scores.parquet`, `textual_index.parquet`, `master_scores_2024.parquet`

In [4]:
t0 = time.time()

if not (PROCESSED_DIR / "sentiment_scores.parquet").exists():
    from src.nlp.collector import run as run_collector
    from src.nlp.sentiment import run as run_sentiment
    from src.nlp.textual_index import run as run_textual
    run_collector()
    run_sentiment()
    run_textual()
    print("NLP pipeline executed.")
else:
    print("NLP parquets already exist.")

for f in ["sentiment_scores", "textual_index", "master_scores_2024"]:
    df = pd.read_parquet(PROCESSED_DIR / f"{f}.parquet")
    print(f"  {f}: {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nAula 4 concluida em {time.time()-t0:.1f}s")

NLP parquets already exist.
  sentiment_scores: 10 rows x 10 cols
  textual_index: 10 rows x 6 cols
  master_scores_2024: 10 rows x 20 cols

Aula 4 concluida em 0.0s


## Aula 5: Modelos de IA

Walk-forward supervisionado (outperform vs Ibovespa) + KMeans clustering.

**Saidas:** `feature_matrix.parquet`, `feature_importance.parquet`, `supervised_predictions_2024.parquet`, `clusters_2024.parquet`, `master_scores_final.parquet`

In [5]:
t0 = time.time()

from src.models.features import build_feature_matrix
from src.models.supervised import run_supervised_pipeline
from src.models.unsupervised import run_unsupervised_pipeline

fm = build_feature_matrix()
sup = run_supervised_pipeline()
unsup = run_unsupervised_pipeline()

# Update master_scores_final with ML signals
master = pd.read_parquet(PROCESSED_DIR / "master_scores_2024.parquet")
preds = pd.read_parquet(PROCESSED_DIR / "supervised_predictions_2024.parquet")
clusters = pd.read_parquet(PROCESSED_DIR / "clusters_2024.parquet")
final = master.merge(preds[["ticker","outperform_probability","label_outperform_pred"]], on="ticker", how="left")
final = final.merge(clusters[["ticker","cluster_label"]], on="ticker", how="left")
final.to_parquet(PROCESSED_DIR / "master_scores_final.parquet", index=False)

for f in ["feature_matrix", "supervised_predictions_2024", "clusters_2024", "master_scores_final"]:
    df = pd.read_parquet(PROCESSED_DIR / f"{f}.parquet")
    print(f"  {f}: {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nAula 5 concluida em {time.time()-t0:.1f}s")

Feature matrix saved: C:\Users\João Paulo\equity_research\data\processed\feature_matrix.parquet  shape=(50, 23)
TRILHA A — Supervised Learning Pipeline

--- Walk-Forward Validation ---
Walk-forward years available: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


 fold                                      train_years  test_year              model  n_train  n_test  accuracy  precision  recall   f1  roc_auc
    1                                 [np.int64(2020)]       2021 LogisticRegression       10      10      0.80       0.88    0.88 0.88     0.88
    1                                 [np.int64(2020)]       2021       RandomForest       10      10      0.70       0.86    0.75 0.80     0.81
    1                                 [np.int64(2020)]       2021   GradientBoosting       10      10      0.80       0.88    0.88 0.88     0.81
    2                 [np.int64(2020), np.int64(2021)]       2022 LogisticRegression       20      10      0.70       0.70    1.00 0.82     0.52
    2                 [np.int64(2020), np.int64(2021)]       2022       RandomForest       20      10      0.60       0.67    0.86 0.75     0.43
    2                 [np.int64(2020), np.int64(2021)]       2022   GradientBoosting       20      10      0.60       0.67    0.86


Silhouette scores: {2: 0.24, 3: 0.294, 4: 0.256, 5: 0.191, 6: 0.139}
Best k = 3
PCA explained variance: [0.393 0.297]

--- Cluster Profiles ---
            net_margin  roe  roa  ebit_margin  debt_to_equity  net_debt_to_ebit  current_ratio  cash_to_revenue  revenue_cagr  net_income_cagr  volatility_annualized  max_drawdown  momentum_6m  momentum_12m cluster_label
cluster_id                                                                                                                                                                                                                  
0                 0.10 0.14 0.02         0.15            1.46              3.05           1.05             0.29          0.23             0.12                   0.22         -0.21        -0.06         -0.16          Risk
1                 0.42 0.30 0.08         0.73            1.67              2.86           1.11             0.28         -0.01             0.02                   0.16         -0.16        -0.09

## Aula 6: Valuation

Multiplos (P/L, P/VP, EV/EBITDA), cenarios macro e ultimate score com recomendacoes.

**Saidas:** `multiples_2024.parquet`, `scenarios_2024.parquet`, `ultimate_scores_2024.parquet`

In [6]:
t0 = time.time()

from src.valuation.valuation_score import run_valuation_pipeline

val_results = run_valuation_pipeline()

for f in ["multiples_2024", "scenarios_2024", "ultimate_scores_2024"]:
    df = pd.read_parquet(PROCESSED_DIR / f"{f}.parquet")
    print(f"  {f}: {df.shape[0]} rows x {df.shape[1]} cols")

print(f"\nAula 6 concluida em {time.time()-t0:.1f}s")

VALUATION PIPELINE

--- Step 1: Multiples ---
Multiples saved: C:\Users\João Paulo\equity_research\data\processed\multiples_2024.parquet  shape=(10, 15)

--- Step 2: Scenarios ---


Scenarios saved: C:\Users\João Paulo\equity_research\data\processed\scenarios_2024.parquet  shape=(30, 7)

--- Step 3: Valuation Score ---
Valuation score saved: C:\Users\João Paulo\equity_research\data\processed\valuation_score_2024.parquet  shape=(10, 17)

--- Step 4: Ultimate Scores ---
Ultimate scores saved: C:\Users\João Paulo\equity_research\data\processed\ultimate_scores_2024.parquet  shape=(10, 31)

--- Final Ranking ---
ticker           sector  ultimate_rank  ultimate_score recommendation  final_score  valuation_score  outperform_probability cluster_label
 ITUB4           Bancos              1           78.50            Buy        69.67            80.00                    0.98          Risk
 EGIE3 Energia Elétrica              2           69.80            Buy        69.56            50.50                    0.99       Quality
 CMIG4 Energia Elétrica              3           66.40            Buy        69.75            70.50                    0.52        Growth
 CPFE3 Energia 

  ultimate_scores_2024: 10 rows x 31 cols

Aula 6 concluida em 0.2s


## Aula 7: Agente de IA

Agente autonomo que integra os 6 tools e gera dossier de investimento.

**Saidas:** `data/output/dossier_*.txt`

In [7]:
t0 = time.time()

from src.agent.agent import EquityResearchAgent

agent = EquityResearchAgent(profile="base")
dossier = agent.generate_dossier(["ITUB4", "EGIE3", "CMIG4"])
# Print summary only (full dossier saved to file)
lines = dossier.strip().splitlines()
# Print last section (summary table)
for line in lines[-12:]:
    print(line)

print(f"\nAula 7 concluida em {time.time()-t0:.1f}s")

[13:09:31] Agent started | profile=base | tickers=['ITUB4', 'EGIE3', 'CMIG4']
[13:09:31] Calling get_macro_snapshot()...



[13:09:32] Analyzing ITUB4 — Itaú Unibanco
[13:09:32] Calling get_kpis(ITUB4, 2024)...
[13:09:32] Calling get_price_history(ITUB4)...
[13:09:32] Calling get_sentiment(ITUB4)...
[13:09:32] Calling get_valuation(ITUB4)...
[13:09:32] Calling get_full_score(ITUB4)...

[13:09:32] Analyzing EGIE3 — Engie Brasil
[13:09:32] Calling get_kpis(EGIE3, 2024)...
[13:09:33] Calling get_price_history(EGIE3)...
[13:09:33] Calling get_sentiment(EGIE3)...
[13:09:33] Calling get_valuation(EGIE3)...
[13:09:33] Calling get_full_score(EGIE3)...

[13:09:33] Analyzing CMIG4 — Cemig
[13:09:33] Calling get_kpis(CMIG4, 2024)...
[13:09:33] Calling get_price_history(CMIG4)...
[13:09:33] Calling get_sentiment(CMIG4)...
[13:09:33] Calling get_valuation(CMIG4)...
[13:09:33] Calling get_full_score(CMIG4)...

[13:09:33] Agent finished — 3 tickers analysed

[13:09:33] Dossier saved to C:\Users\João Paulo\equity_research\data\output\dossier_2026-03-27_base.txt
    Profile-adjusted Score: 69.0/100
------------------------

## Aula 8: Relatorio PDF Final

Gera relatorio profissional de 6 paginas com heatmaps, ranking e cenarios.

**Saida:** `data/output/relatorio_final.pdf`

In [8]:
t0 = time.time()

from src.report.report_generator import generate_report

pdf_path = generate_report()
print(f"PDF size: {pdf_path.stat().st_size:,} bytes")

print(f"\nAula 8 concluida em {time.time()-t0:.1f}s")

Report generated: C:\Users\João Paulo\equity_research\data\output\relatorio_final.pdf
PDF size: 205,920 bytes

Aula 8 concluida em 2.1s


## Resumo: Todos os Artefatos

In [9]:
import os

print("=" * 65)
print("ARTEFATOS DO PIPELINE")
print("=" * 65)

# Parquets
print("\ndata/processed/")
total_rows = 0
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    df = pd.read_parquet(f)
    total_rows += df.shape[0]
    print(f"  {f.name:45s} {df.shape[0]:>6,} rows x {df.shape[1]:>2} cols")

print(f"  {'TOTAL':45s} {total_rows:>6,} rows")

# Output files
print("\ndata/output/")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.name:45s} {size:>10,} bytes")

print("\nPipeline completo.")

ARTEFATOS DO PIPELINE

data/processed/
  clusters_2024.parquet                             10 rows x  6 cols
  feature_importance.parquet                        18 rows x  3 cols
  feature_matrix.parquet                            50 rows x 23 cols
  fundamentals_long.parquet                     15,939 rows x  7 cols
  income_long.parquet                               49 rows x  6 cols
  kpis_wide.parquet                                 50 rows x 17 cols
  macro_monthly.parquet                             72 rows x  4 cols
  master_scores_2024.parquet                        10 rows x 20 cols
  master_scores_final.parquet                       10 rows x 23 cols
  multiples_2024.parquet                            10 rows x 15 cols
  prices_daily.parquet                          16,434 rows x  7 cols
  scenarios_2024.parquet                            30 rows x  7 cols
  scores_2024.parquet                               10 rows x 13 cols
  sentiment_scores.parquet                         